# CrashVision — Re-entrenamiento M2: Data Augmentation dirigido
### Sistema de Detección Automática de Accidentes de Tránsito

**Proyecto:** CrashVision — Inteligencia Artificial 1
**Universidad:** UDLA (Universidad de las Américas, Quito)
**Docente:** Prof. Enrique Vinicio Carrera
**Autores:** Sebastian Abad · Daniel Cornejo
**Objetivo M2:** Re-entrenar **solo YOLOv8s** con la receta de M1 (150 epochs, `patience=20`)
**+ data augmentation dirigido** a los modos de fallo del VIDEO real.

---

## Contexto y motivación

M1 ("más épocas + early stopping") **mejoró el test set Roboflow** (F1 0.873) pero
**REGRESÓ en video real**: el recall cayó de **75% (6/8)** a **50% (4/8)** @ conf=0.30.
Conclusión: optimizar la distribución conocida no arregla la generalización.

M2 ataca el cuello de botella real exponiendo al modelo a variaciones que aproximan el
video de producción (iluminación/noche, clima, ángulo dashcam vs. vigilancia, distancia,
oclusión) vía augmentation, **manteniendo la receta de M1**. Solo se entrena `s`
(YOLOv8n se descartó en M1 por regresión de recall).

| Parámetro | M1 | M2 (este notebook) |
|---|---|---|
| `epochs` | 150 | 150 (sin cambio) |
| `patience` | 20 | 20 (sin cambio) |
| `batch` / `imgsz` | 16 / 640 | 16 / 640 (sin cambio) |
| Modelos | n + s | **solo s** |
| Augmentation | default | **dirigido** (hsv_*, geometría, mixup, copy_paste, erasing) |

> ⚖️ **Criterio de entrega:** la métrica que MANDA es el **recall en video real** (batir el
> baseline 75%), no la del test set Roboflow. La tabla del test set (Sección 6) es
> secundaria; el veredicto final sale de `tools/calibrate_threshold.py --weights` sobre
> los 8 videos, en el repo `ProjectCrashIA1`.

---

## Contenido del notebook

| Sección | Descripción |
|---|---|
| 0 | Verificación del entorno (GPU, CUDA, RAM) |
| 1 | Configuración de rutas locales M2 |
| 2 | Dependencias e imports |
| 3 | Descarga del dataset Roboflow v2 |
| 4 | Re-entrenamiento YOLOv8s M2 (150 epochs + augmentation dirigido) |
| 5 | Evaluación comparativa Baseline vs. M2 en test set |
| 6 | Visualización de curvas de entrenamiento |
| 7 | Gráfica comparativa Baseline vs. M2 |
| 8 | Exportación del modelo y artefactos |
| 9 | Resumen final y siguiente paso (validación en 8 videos) |


---
## 0. Verificación del entorno

In [1]:
import os
# Forzar GPU 0 antes de importar torch — evita que VS Code/Jupyter inyecte CUDA_VISIBLE_DEVICES=""
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import subprocess, sys
import torch

print("=" * 55)
print("VERIFICACIÓN DE ENTORNO — CrashVision M2 (Local)")
print("=" * 55)

# GPU
try:
    gpu_info = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
         '--format=csv,noheader'],
        encoding='utf-8').strip()
    print(f"\nGPU detectada:")
    for line in gpu_info.split('\n'):
        print(f" {line}")
except FileNotFoundError:
    print("\nGPU (nvidia-smi) NO detectada.")

# RAM
import psutil
ram = psutil.virtual_memory()
print(f"\nRAM disponible: {ram.available / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB total")

# Python
print(f"\nPython: {sys.version.split()[0]}")

# CUDA via PyTorch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Dispositivo CUDA: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM total: {props.total_memory / 1e9:.1f} GB")
else:
    print("ADVERTENCIA: CUDA no detectado. M2 en CPU es inviable (horas). Revisa el kernel.")

print("\n" + "=" * 55)

VERIFICACIÓN DE ENTORNO — CrashVision M2 (Local)

GPU detectada:
 NVIDIA GeForce RTX 4060 Laptop GPU, 8188 MiB, 610.47

RAM disponible: 11.8 GB / 34.0 GB total

Python: 3.11.9
PyTorch: 2.11.0+cu128
CUDA disponible: True
Dispositivo CUDA: NVIDIA GeForce RTX 4060 Laptop GPU
   VRAM total: 8.6 GB



---
## 1. Configuración de rutas locales M2

Mismo esquema que M1 (`x\CrashVision\...`), con variables `*_M2` y carpeta `m2`. Solo `s`.

In [2]:
import os
from pathlib import Path

# Carpeta raíz del proyecto (misma ubicación que este notebook, laptop de entrenamiento)
BASE_DIR          = Path(r"C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x")
DRIVE_BASE        = str(BASE_DIR / "CrashVision")
DRIVE_MODELS_BASE = str(BASE_DIR / "CrashVision" / "models")
DRIVE_MODELS_M2   = str(BASE_DIR / "CrashVision" / "models" / "m2")
DRIVE_RESULTS_S   = str(BASE_DIR / "CrashVision" / "results" / "yolov8s_m2")
DRIVE_COMPARE_M2  = str(BASE_DIR / "CrashVision" / "results" / "comparison_m2")

for folder in [DRIVE_MODELS_M2, DRIVE_RESULTS_S, DRIVE_COMPARE_M2]:
    os.makedirs(folder, exist_ok=True)
    print(f"Carpeta lista: {folder}")

# Verificar que el baseline 's' existe (necesario para la comparación de la Sección 5)
path_base_s = os.path.join(DRIVE_MODELS_BASE, 'best_s.pt')
if os.path.exists(path_base_s):
    print(f"Baseline encontrado: best_s.pt ({os.path.getsize(path_base_s) / 1e6:.1f} MB)")
else:
    print(f"Baseline NO encontrado: {path_base_s}")
    print("La Sección 5 necesita best_s.pt para la comparación Baseline vs M2.")

print("\nEstructura local:")
print(f"  {DRIVE_BASE}/")
print("  ├── models/")
print("  │   ├── best_s.pt        ← baseline original (NO se sobreescribe)")
print("  │   └── m2/")
print("  │       └── best_s_m2.pt ← modelo s re-entrenado con augmentation")
print("  └── results/")
print("      ├── yolov8s_m2/      ← curvas y métricas s M2")
print("      └── comparison_m2/   ← tabla comparativa Baseline vs M2")

Carpeta lista: C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\m2
Carpeta lista: C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\yolov8s_m2
Carpeta lista: C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\comparison_m2
Baseline NO encontrado: C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\best_s.pt
La Sección 5 necesita best_s.pt para la comparación Baseline vs M2.

Estructura local:
  C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision/
  ├── models/
  │   ├── best_s.pt        ← baseline original (NO se sobreescribe)
  │   └── m2/
  │       └── best_s_m2.pt ← modelo s re-entrenado con augmentation
  └── results/
      ├── yolov8s_m2/      ← curvas y métricas s M2
      └── comparison_m2/   ← tabla comparativa Baseline vs M2


---
## 2. Dependencias e imports

In [3]:
import ultralytics
import roboflow
print(f"Ultralytics: {ultralytics.__version__}")
print(f"Roboflow: {roboflow.__version__}")

from ultralytics import YOLO
print("YOLOv8 importado correctamente")

import os, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
print("Librerías auxiliares listas")

Ultralytics: 8.4.75
Roboflow: 1.3.10
YOLOv8 importado correctamente
Librerías auxiliares listas


---
## 3. Descarga del dataset Roboflow v2

Mismo dataset y distribución que M1 — lo único que cambia en M2 es el augmentation.

In [4]:
from roboflow import Roboflow
from pathlib import Path
import yaml

ROBOFLOW_API_KEY = "DwCmQ00XOYIzR0pfSFUK"
WORKSPACE_SLUG = "accident-detection-model"
PROJECT_SLUG = "accident-detection-model"
DATASET_VERSION = 2

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_SLUG).project(PROJECT_SLUG)
dataset = project.version(DATASET_VERSION).download("yolov8")

DATASET_PATH = Path(dataset.location)
DATA_YAML    = str(DATASET_PATH / 'data.yaml')

print(f"\nDataset descargado en: {DATASET_PATH}")

# Validar data.yaml
with open(DATA_YAML, 'r') as f:
    yaml_content = yaml.safe_load(f)

names_lower = [n.lower() for n in yaml_content.get('names', [])]
assert yaml_content.get('nc') == 1, "Se esperaba nc=1"
assert 'accident' in names_lower, "Clase 'accident' no encontrada"
print(f"data.yaml válido — nc={yaml_content['nc']}, clase='{yaml_content['names'][0]}'")
print(f"data.yaml → {DATA_YAML}")

loading Roboflow workspace...
loading Roboflow project...

Dataset descargado en: c:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\Accident-detection-model-2
data.yaml válido — nc=1, clase='Accident'
data.yaml → c:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\Accident-detection-model-2\data.yaml


---
## 4. Re-entrenamiento YOLOv8s M2 (150 epochs + augmentation dirigido)

Receta heredada de M1 (150 epochs, `patience=20`, batch=16, imgsz=640, base COCO `yolov8s.pt`).
**Lo único nuevo es el bloque de augmentation**, dirigido a los modos de fallo del video real:
- **Iluminación/clima** (noche, baja luz, días grises, lluvia): `hsv_v=0.60`, `hsv_s=0.80`, `hsv_h=0.02`.
- **Geometría / punto de vista** (dashcam vs. vigilancia elevada): `degrees=10`, `translate=0.15`, `scale=0.6`, `shear=3`, `perspective=0.0005`.
- **Composición y oclusión**: `mixup=0.15`, `copy_paste=0.10`, `erasing=0.4`, `close_mosaic=10`.

> ✅ **Este entrenamiento YA se corrió en la laptop** (vía el `train_m2.py` anterior):
> early stopping en **epoch 54/150** (mejor modelo en **epoch 34**), ~30 min, y los pesos
> quedaron en `runs/detect/crashvision_s_m2/weights/best.pt`. Por eso la celda de abajo es
> **idempotente**: con `FORCE_RETRAIN=False` detecta ese run y **no re-entrena** (evita
> sobreescribirlo y perder 30 min) — solo lee `results.csv` para reportar la convergencia.
> Las Secciones 5–9 (eval en test set, CSVs, curvas y export) son justo lo que el `.py` nunca
> generó. Pon `FORCE_RETRAIN=True` solo si quieres re-entrenar desde cero.

In [5]:
from ultralytics import YOLO
import torch, time

# Hiperparámetros M2 — heredados de M1 (solo 's')
EPOCHS_M2    = 150
PATIENCE_M2  = 20
BATCH_SIZE   = 16
IMG_SIZE     = 640
DEVICE       = 0  # GPU — RTX 4060 Laptop
MODEL_NAME_S = 'crashvision_s_m2'

# El entrenamiento M2 ya se ejecutó (early stopping epoch 54, best epoch 34, ~30 min).
# FORCE_RETRAIN=False -> NO re-entrena si el run ya existe (no sobreescribe los pesos).
FORCE_RETRAIN = False

run_dir = Path(f'runs/detect/{MODEL_NAME_S}')
best_pt = run_dir / 'weights' / 'best.pt'

if best_pt.exists() and not FORCE_RETRAIN:
    print("=" * 58)
    print("  RUN M2 YA EXISTE — se OMITE el entrenamiento")
    print("=" * 58)
    print(f"  Pesos: {best_pt}  ({best_pt.stat().st_size / 1e6:.1f} MB)")
    print("  FORCE_RETRAIN=False → reutiliza estos pesos. Continúa con la Sección 5.")
    print("  (Pon FORCE_RETRAIN=True si quieres re-entrenar desde cero.)")
else:
    print("=" * 58)
    print("  RE-ENTRENAMIENTO YOLOv8s M2 — CrashVision (augmentation dirigido)")
    print("=" * 58)
    print(f"  epochs={EPOCHS_M2} | patience={PATIENCE_M2} | batch={BATCH_SIZE} | imgsz={IMG_SIZE}")
    print(f"  device={DEVICE} | name={MODEL_NAME_S}")
    print("=" * 58)

    model_s_m2 = YOLO('yolov8s.pt')  # base COCO, entrenamiento desde cero (igual que M1)

    t_start_s = time.time()
    results_s_m2 = model_s_m2.train(
        data=DATA_YAML,
        epochs=EPOCHS_M2,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        patience=PATIENCE_M2,
        device=DEVICE,
        name=MODEL_NAME_S,
        exist_ok=True,
        workers=2,
        cache=False,
        save=True,
        plots=True,
        verbose=True,
        # --- M2: AUGMENTATION dirigido a los modos de fallo del VIDEO real ---
        # Iluminación / clima (noche, baja luz, días grises, lluvia):
        hsv_h=0.020,        # tono   (default 0.015)
        hsv_s=0.80,         # satur. (default 0.70)
        hsv_v=0.60,         # brillo (default 0.40)  <- clave para noche/baja luz
        # Geometría / punto de vista (dashcam vs. cámara de vigilancia elevada):
        degrees=10.0,       # rotación         (default 0.0)
        translate=0.15,     # desplazamiento   (default 0.10)
        scale=0.6,          # escala/distancia (default 0.50)
        shear=3.0,          # cizalla          (default 0.0)
        perspective=0.0005, # perspectiva      (default 0.0)
        fliplr=0.5,         # espejo horizontal(default 0.5)
        # Composición y oclusión:
        mosaic=1.0,         # (default 1.0)
        close_mosaic=10,    # apaga mosaic las últimas 10 epochs para estabilizar
        mixup=0.15,         # mezcla de imágenes      (default 0.0)
        copy_paste=0.10,    # copiar-pegar instancias (default 0.0)
        erasing=0.4,        # random erasing/oclusión (default 0.4)
    )
    print(f"\nYOLOv8s M2 completado en {(time.time() - t_start_s)/60:.1f} minutos")

# --- Reporte de convergencia desde results.csv (corriera ahora o antes) ------
def _find_col(cols, needle):
    needle = needle.lower().replace(' ', '')
    for c in cols:
        if needle in c.lower().replace(' ', ''):
            return c
    return None

csv_s = run_dir / 'results.csv'
if csv_s.exists():
    df_s_check = pd.read_csv(str(csv_s))
    df_s_check.columns = df_s_check.columns.str.strip()
    n_ep = len(df_s_check)

    # Epoch óptimo = máximo de la "fitness" de Ultralytics (0.1*mAP50 + 0.9*mAP50-95)
    c50 = _find_col(df_s_check.columns, 'map50(b)')
    c95 = _find_col(df_s_check.columns, 'map50-95(b)')
    best_line = ''
    if c50 and c95:
        fitness = 0.1 * df_s_check[c50] + 0.9 * df_s_check[c95]
        best_i = int(fitness.idxmax())
        best_epoch = int(df_s_check.iloc[best_i].get('epoch', best_i + 1))
        best_line = (f"  Mejor epoch ≈ {best_epoch} "
                     f"(mAP50={df_s_check[c50].iloc[best_i]:.3f}, "
                     f"mAP50-95={df_s_check[c95].iloc[best_i]:.3f}) → es el que quedó en best.pt")

    if n_ep < EPOCHS_M2:
        print(f"\nEarly stopping activado en epoch {n_ep} / {EPOCHS_M2}  ← modelo convergió")
    else:
        print(f"\nEl modelo completó las {EPOCHS_M2} epochs SIN early stopping.")
        print("El augmentation puede requerir más epochs; revisa las curvas de loss.")
    if best_line:
        print(best_line)

  RUN M2 YA EXISTE — se OMITE el entrenamiento
  Pesos: runs\detect\crashvision_s_m2\weights\best.pt  (22.5 MB)
  FORCE_RETRAIN=False → reutiliza estos pesos. Continúa con la Sección 5.
  (Pon FORCE_RETRAIN=True si quieres re-entrenar desde cero.)

Early stopping activado en epoch 54 / 150  ← modelo convergió
  Mejor epoch ≈ 34 (mAP50=0.700, mAP50-95=0.304) → es el que quedó en best.pt


---
## 5. Evaluación comparativa Baseline vs. M2 (test set, conf=0.30)

Solo `s`: `best_s.pt` (baseline) vs. M2. **Recordatorio:** el test set Roboflow es la
distribución conocida; una posible baja aquí NO condena a M2 — el veredicto real es el
recall en los 8 videos (Sección 9).

In [6]:
from ultralytics import YOLO

CONF_EVAL = 0.30  # threshold óptimo — Experimento 1

# Solo modelos 's': baseline vs M2
models_eval = {
    'YOLOv8s Baseline': f'{DRIVE_MODELS_BASE}/best_s.pt',
    'YOLOv8s M2':       f'runs/detect/{MODEL_NAME_S}/weights/best.pt',
}

for name, path in models_eval.items():
    if os.path.exists(path):
        print(f"{name:<22} → {os.path.getsize(path) / 1e6:.1f} MB")
    else:
        print(f"NO encontrado: {name} → {path}")

print(f"\nEvaluando en test set (conf={CONF_EVAL})...")
print("-" * 65)

comparison_results = []

for model_name, model_path in models_eval.items():
    if not os.path.exists(model_path):
        print(f"Saltando {model_name} (archivo no encontrado)")
        continue

    print(f"  Evaluando {model_name}...", end=' ', flush=True)
    m_obj = YOLO(model_path)
    m = m_obj.val(
        data=DATA_YAML,
        split='test',
        conf=CONF_EVAL,
        iou=0.50,
        verbose=False,
    )
    precision = m.box.mp
    recall    = m.box.mr
    map50     = m.box.map50
    map_full  = m.box.map
    f1        = 2 * (precision * recall) / (precision + recall + 1e-9)
    size_mb   = os.path.getsize(model_path) / 1e6

    comparison_results.append({
        'Modelo':     model_name,
        'Precision':  round(precision, 4),
        'Recall':     round(recall, 4),
        'F1-score':   round(f1, 4),
        'mAP@50':     round(map50, 4),
        'mAP@50:95':  round(map_full, 4),
        'Tamaño(MB)': round(size_mb, 1),
    })
    print(f"P={precision:.3f} | R={recall:.3f} | F1={f1:.3f} | mAP@50={map50:.3f}")

df_comparison = pd.DataFrame(comparison_results)

print("\n" + "=" * 75)
print("  COMPARACIÓN BASELINE vs M2 (conf=0.30, test set)")
print("=" * 75)
print(df_comparison.to_string(index=False))

# Análisis automático de mejora (Baseline s -> M2 s)
print("\n── Análisis de mejora (test set) ──")
base_rows = df_comparison[df_comparison['Modelo'] == 'YOLOv8s Baseline']
m2_rows   = df_comparison[df_comparison['Modelo'] == 'YOLOv8s M2']
if not base_rows.empty and not m2_rows.empty:
    base_row = base_rows.iloc[0]
    m2_row   = m2_rows.iloc[0]
    sign = lambda v: f"+{v:.4f}" if v >= 0 else f"{v:.4f}"
    delta_f1    = m2_row['F1-score'] - base_row['F1-score']
    delta_map50 = m2_row['mAP@50']   - base_row['mAP@50']
    delta_rec   = m2_row['Recall']   - base_row['Recall']
    print(f"  YOLOv8s M2      : ΔF1={sign(delta_f1)} | ΔmAP@50={sign(delta_map50)} | ΔRecall={sign(delta_rec)}")
    print("  (Nota: la métrica que decide la entrega es el recall en VIDEO real, no ésta.)")

# Guardar CSV
csv_out = f'{DRIVE_COMPARE_M2}/m2_vs_baseline.csv'
df_comparison.to_csv(csv_out, index=False)
print(f"\nm2_vs_baseline.csv guardado en {csv_out}")

NO encontrado: YOLOv8s Baseline → C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models/best_s.pt
YOLOv8s M2             → 22.5 MB

Evaluando en test set (conf=0.3)...
-----------------------------------------------------------------
Saltando YOLOv8s Baseline (archivo no encontrado)
  Evaluando YOLOv8s M2... Ultralytics 8.4.75  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 142.158.7 MB/s, size: 64.2 KB)
val: Scanning C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\Accident-detection-model-2\test\labels.cache... 362 images, 186 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 362/362  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 6.3it/s 3.7s0.1s
                   all        362        204      0.797      0.711      0.675      0.298


---
## 6. Curvas de entrenamiento — YOLOv8s M2

In [7]:
# helper
def safe_plot(ax, df, col_name, title, color, ylabel=''):
    """Grafica una columna si existe, con matching parcial de nombre."""
    match = [c for c in df.columns if col_name.lower() in c.lower()]
    if match:
        epoch_col = 'epoch' if 'epoch' in df.columns else df.columns[0]
        ax.plot(df[epoch_col], df[match[0]], color=color, linewidth=1.8)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel or title)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(left=0)
    else:
        ax.text(0.5, 0.5, f'No disponible\n({col_name})', ha='center', va='center',
                transform=ax.transAxes, color='gray')
        ax.set_title(title, fontsize=10)


# Curvas YOLOv8s M2
csv_s_m2 = f'runs/detect/{MODEL_NAME_S}/results.csv'
if os.path.exists(csv_s_m2):
    df_s = pd.read_csv(csv_s_m2)
    df_s.columns = df_s.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f'Curvas de Entrenamiento — YOLOv8s M2 ({len(df_s)} epochs)', fontsize=14, fontweight='bold')

    safe_plot(axes[0,0], df_s, 'train/box_loss',   'Box Loss (train)',    '#E74C3C', 'Loss')
    safe_plot(axes[0,1], df_s, 'train/cls_loss',   'Class Loss (train)',  '#E67E22', 'Loss')
    safe_plot(axes[0,2], df_s, 'train/dfl_loss',   'DFL Loss (train)',    '#F39C12', 'Loss')
    safe_plot(axes[1,0], df_s, 'metrics/mAP50',    'mAP@50 (val)',        '#27AE60', 'mAP')
    safe_plot(axes[1,1], df_s, 'metrics/precision','Precision (val)',      '#2980B9', 'Precision')
    safe_plot(axes[1,2], df_s, 'metrics/recall',   'Recall (val)',         '#8E44AD', 'Recall')

    plt.tight_layout()
    plt.savefig(f'{DRIVE_RESULTS_S}/training_curves_m2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Curvas YOLOv8s M2 guardadas en", DRIVE_RESULTS_S)
else:
    print(f"results.csv no encontrado: {csv_s_m2}")

<Figure size 1600x900 with 6 Axes>

Curvas YOLOv8s M2 guardadas en C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\yolov8s_m2


---
## 7. Gráfica comparativa Baseline vs. M2

In [8]:
# Gráfica comparativa Baseline vs M2
if len(df_comparison) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 6))
    fig.suptitle('Baseline vs M2 — CrashVision (conf=0.30, test set)', fontsize=14, fontweight='bold')

    model_labels = df_comparison['Modelo'].tolist()
    x = np.arange(len(model_labels))
    colors = ['#95A5A6', '#E74C3C'][:len(model_labels)]  # gris=baseline, rojo=M2

    # Subplot 1: F1 y mAP@50
    ax1 = axes[0]
    w = 0.3
    f1_vals    = df_comparison['F1-score'].tolist()
    map50_vals = df_comparison['mAP@50'].tolist()
    b1 = ax1.bar(x - w/2, f1_vals,    w, label='F1-score', color='#2980B9', alpha=0.85)
    b2 = ax1.bar(x + w/2, map50_vals, w, label='mAP@50',   color='#27AE60', alpha=0.85)
    for bar, v in list(zip(b1, f1_vals)) + list(zip(b2, map50_vals)):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax1.set_xticks(x); ax1.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=8)
    ax1.set_ylim(0, 1.05); ax1.set_title('F1-score y mAP@50'); ax1.set_ylabel('Score')
    ax1.legend(); ax1.grid(axis='y', alpha=0.3)

    # Subplot 2: Precision y Recall
    ax2 = axes[1]
    prec_vals = df_comparison['Precision'].tolist()
    rec_vals  = df_comparison['Recall'].tolist()
    b3 = ax2.bar(x - w/2, prec_vals, w, label='Precision', color='#E67E22', alpha=0.85)
    b4 = ax2.bar(x + w/2, rec_vals,  w, label='Recall',    color='#8E44AD', alpha=0.85)
    for bar, v in list(zip(b3, prec_vals)) + list(zip(b4, rec_vals)):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax2.set_xticks(x); ax2.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=8)
    ax2.set_ylim(0, 1.05); ax2.set_title('Precision y Recall'); ax2.set_ylabel('Score')
    ax2.legend(); ax2.grid(axis='y', alpha=0.3)

    # Subplot 3: mAP@50:95
    ax3 = axes[2]
    map_full_vals = df_comparison['mAP@50:95'].tolist()
    bars = ax3.bar(model_labels, map_full_vals, color=colors, alpha=0.85, width=0.5)
    for bar, v in zip(bars, map_full_vals):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax3.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=8)
    ax3.set_ylim(0, 0.7); ax3.set_title('mAP@50:95'); ax3.set_ylabel('mAP@50:95')
    ax3.grid(axis='y', alpha=0.3)

    # Línea de referencia (baseline YOLOv8s mAP@50 = 0.8278)
    for ax in [ax1, ax2]:
        ax.axhline(y=0.8278, color='gray', linestyle='--', linewidth=1, alpha=0.6)

    plt.tight_layout()
    plt.savefig(f'{DRIVE_COMPARE_M2}/m2_vs_baseline_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Gráfica comparativa guardada en", DRIVE_COMPARE_M2)

---
## 8. Exportación del modelo y artefactos

Copia `best_s_m2.pt`/`last_s_m2.pt` a `models/m2/` y los artefactos de `runs/` a `results/yolov8s_m2/`.

In [9]:
import shutil, os

best_src = f'runs/detect/{MODEL_NAME_S}/weights/best.pt'
best_dst = os.path.join(DRIVE_MODELS_M2, 'best_s_m2.pt')
last_src = f'runs/detect/{MODEL_NAME_S}/weights/last.pt'
last_dst = os.path.join(DRIVE_MODELS_M2, 'last_s_m2.pt')

print("=" * 58)
print("EXPORTANDO MODELO M2 A CARPETA LOCAL")
print("=" * 58)

if not os.path.exists(best_src):
    print(f"YOLOv8s M2: best.pt no encontrado en {best_src}")
else:
    shutil.copy(best_src, best_dst)
    if os.path.exists(last_src):
        shutil.copy(last_src, last_dst)

    size_mb = os.path.getsize(best_dst) / 1e6
    print(f"\nYOLOv8s M2")
    print(f"{best_dst} ({size_mb:.1f} MB)")

    results_dir = Path(f'runs/detect/{MODEL_NAME_S}')
    artifacts   = [
        'results.csv',
        'confusion_matrix.png',
        'BoxPR_curve.png', 'PR_curve.png',
        'BoxF1_curve.png', 'F1_curve.png',
        'results.png',
    ]
    for fname in artifacts:
        src = results_dir / fname
        if src.exists():
            shutil.copy(str(src), os.path.join(DRIVE_RESULTS_S, fname))
            print(f"  {fname}")

print("\n" + "=" * 58)
print("  SIGUIENTE PASO — validar en VIDEO real (repo ProjectCrashIA1):")
print("  1) Copia best_s_m2.pt a  ProjectCrashIA1/models/best_s_m2.pt")
print("  2) venv/Scripts/python tools/calibrate_threshold.py \\")
print("       --weights models/best_s_m2.pt --labels tools/labels.csv")
print("  La entrega se decide por el recall en los 8 videos (batir 75%).")

EXPORTANDO MODELO M2 A CARPETA LOCAL

YOLOv8s M2
C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\m2\best_s_m2.pt (22.5 MB)
  results.csv
  confusion_matrix.png
  BoxPR_curve.png
  BoxF1_curve.png
  results.png

  SIGUIENTE PASO — validar en VIDEO real (repo ProjectCrashIA1):
  1) Copia best_s_m2.pt a  ProjectCrashIA1/models/best_s_m2.pt
  2) venv/Scripts/python tools/calibrate_threshold.py \
       --weights models/best_s_m2.pt --labels tools/labels.csv
  La entrega se decide por el recall en los 8 videos (batir 75%).


---
## 9. Resumen final y siguiente paso

In [10]:
print("\n" + "=" * 68)
print("  RESUMEN FINAL — CrashVision M2")
print("=" * 68)

print("\nEarly stopping:")
if os.path.exists(csv_s_m2):
    df_tmp = pd.read_csv(csv_s_m2)
    n_ep = len(df_tmp)
    status = f"paró en epoch {n_ep} (convergió)" if n_ep < EPOCHS_M2 else f"completó {n_ep} epochs sin parar"
    print(f"   YOLOv8s M2        : {status}")

if not df_comparison.empty:
    print("\nComparación Baseline vs M2 (conf=0.30, test set):")
    print(df_comparison[['Modelo','Precision','Recall','F1-score','mAP@50','mAP@50:95']].to_string(index=False))

print("\nLectura del test set (NO es el criterio de entrega):")
base_rows = df_comparison[df_comparison['Modelo'] == 'YOLOv8s Baseline']
m2_rows   = df_comparison[df_comparison['Modelo'] == 'YOLOv8s M2']
if not base_rows.empty and not m2_rows.empty:
    base = base_rows.iloc[0]; m2 = m2_rows.iloc[0]
    if m2['F1-score'] >= base['F1-score']:
        print(f"   M2 mantiene/mejora F1 en test (Δ={m2['F1-score']-base['F1-score']:+.4f}).")
    else:
        print(f"   M2 baja F1 en test (Δ={m2['F1-score']-base['F1-score']:+.4f}) — esperable con augmentation;")
        print("   NO condena a M2. Lo que importa es el recall en video real.")

print("\n⚖️  DECISIÓN DE ENTREGA — se toma en VIDEO real, no aquí:")
print("   En el repo ProjectCrashIA1:")
print("     venv/Scripts/python tools/calibrate_threshold.py --weights models/best_s_m2.pt \\")
print("        --labels tools/labels.csv")
print("   Baseline a batir: recall 75% (6/8). M1 regresó a 50% (4/8).")
print("   - Si M2 > 75%  → nuevo ganador; recién ahí promover a config.py + TRAINING_METRICS.")
print("   - Si M2 ≤ 75%  → mantener best_s.pt; documentar y considerar M3 (dataset + YOLOv8m).")

print("\nArchivos generados localmente:")
for f in [
    os.path.join(DRIVE_MODELS_M2, 'best_s_m2.pt'),
    os.path.join(DRIVE_RESULTS_S, 'results.csv'),
    os.path.join(DRIVE_RESULTS_S, 'training_curves_m2.png'),
    os.path.join(DRIVE_COMPARE_M2, 'm2_vs_baseline.csv'),
    os.path.join(DRIVE_COMPARE_M2, 'm2_vs_baseline_comparison.png'),
]:
    print(f"  {f}")
print("\n" + "=" * 68)


  RESUMEN FINAL — CrashVision M2

Early stopping:
   YOLOv8s M2        : paró en epoch 54 (convergió)

Comparación Baseline vs M2 (conf=0.30, test set):
    Modelo  Precision  Recall  F1-score  mAP@50  mAP@50:95
YOLOv8s M2     0.7967  0.7108    0.7513   0.675     0.2983

Lectura del test set (NO es el criterio de entrega):

⚖️  DECISIÓN DE ENTREGA — se toma en VIDEO real, no aquí:
   En el repo ProjectCrashIA1:
     venv/Scripts/python tools/calibrate_threshold.py --weights models/best_s_m2.pt \
        --labels tools/labels.csv
   Baseline a batir: recall 75% (6/8). M1 regresó a 50% (4/8).
   - Si M2 > 75%  → nuevo ganador; recién ahí promover a config.py + TRAINING_METRICS.
   - Si M2 ≤ 75%  → mantener best_s.pt; documentar y considerar M3 (dataset + YOLOv8m).

Archivos generados localmente:
  C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\m2\best_s_m2.pt
  C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\yolov8s_m2\results.csv
  C:\Use